# Multi-block final_sync_df inspection

Inspect `final_sync_df.csv` for **several blocks** of a single animal in one run. Set the parameters below, run the notebook, and get one explorable Bokeh plot per block in tabs (pan, zoom, reset per tab).

- **Parameters**: `experiment_path`, `animal_call`, `block_numbers` (list).
- **Per block**: Loads `analysis/final_sync_df.csv` (or `blocksync_df.csv`), optionally parses OE events for LED TTL verticals, and builds the same sanity plot used in block synchronization.
- **Output**: One tab per block; each tab shows left/right eye brightness vs OE time with optional LED rising (green) and falling (red) verticals.

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
from __future__ import annotations
from pathlib import Path

from bokeh.io import output_notebook, show
from bokeh.models import Tabs, TabPanel

from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.block_sync_core import load_final_sync_df
from eye_tracking_system_tools.preprocessing.block_sync_visualization import make_final_df_sanity_figure

output_notebook()

## Parameters

Set **experiment_path**, **animal_call**, and **block_numbers** (list). Optionally set **channeldict_by_animal** so LED TTL verticals can be drawn (requires `parse_open_ephys_events` to run).

In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal_call = "PV_126"
block_numbers = [6, 7]  # list of block numbers for this animal
show_led = True         # draw LED rising/falling verticals when oe_events available
bad_blocks = []

# Channel mapping (line -> role). Used so parse_open_ephys_events can fill oe_events for LED verticals.
channeldict_by_animal = {
    "PV_208": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
    "TE_21": {1: "Arena_TTL", 4: "LED_driver", 5: "R_eye_TTL", 8: "L_eye_TTL"},
    "PV_106": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
    "PV_126": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
}
channeldict = channeldict_by_animal.get(animal_call)

## Load blocks and build explorable Bokeh tabs

For each block: load `final_sync_df`, optionally parse OE events for LED, build one Bokeh figure per block, then show all in tabs.

In [ ]:
block_collection = uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal_call,
    bad_blocks=bad_blocks,
)

if not block_collection:
    raise ValueError(
        f"No blocks found for animal={animal_call!r}, block_numbers={block_numbers!r} under {experiment_path}"
    )

panels = []
for block in block_collection:
    if channeldict is not None:
        block.channeldict = channeldict
    load_final_sync_df(block, verbose=True)
    if show_led and channeldict is not None:
        try:
            block.parse_open_ephys_events(overwrite=False)
        except Exception as e:
            print(f"[Block {block.block_num}] Could not parse OE events for LED: {e}")
    fs = block.sample_rate if hasattr(block, "sample_rate") else block.get_sample_rate()
    title = f"{animal_call} block_{block.block_num} final_sync_df"
    p = make_final_df_sanity_figure(
        block.final_sync_df, fs, show_led=show_led, block=block, title=title
    )
    panels.append(TabPanel(child=p, title=f"Block {block.block_num}"))

tabs = Tabs(tabs=panels)
show(tabs)